In [0]:
%pip install google-genai

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys
import json
import shutil
import pandas as pd
from pyspark import StorageLevel

import os
import json
import shutil
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from google import genai

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_FOLDER = Path("/Volumes/workspace/default/py_de_ai_project")

LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"

PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"

for folder in [LANDING, PROCESSING, ARCHIVE, ERROR, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. GEMINI CONFIGURATION
# ============================================================

load_dotenv(PROJECT_FOLDER / ".env")

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

MODEL = "gemini-3.5-flash-lite"


# ============================================================
# 3. SPARK DATAFRAME INGESTION
# ============================================================

def load_dataframe(file_path):

    extension = file_path.suffix.lower()
    file_path = str(file_path)

    if extension == ".csv":

        return (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("mode", "FAILFAST")
            .csv(file_path)
        )

    elif extension == ".json":

        return (
            spark.read
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .json(file_path)
        )

    elif extension == ".txt":

        return (
            spark.read
            .option("header", "true")
            .option("sep", "|")
            .option("inferSchema", "true")
            .option("mode", "FAILFAST")
            .csv(file_path)
        )

    elif extension == ".xlsx":

        # Excel requires an additional Spark connector
        # or a separate Excel reader.
        # Here Pandas is used only for ingestion.

        pandas_df = pd.read_excel(file_path)

        return spark.createDataFrame(pandas_df)

    else:

        print("Unsupported file type:", extension)
        return None


# ============================================================
# 4. SPARK DATA PROFILING
# ============================================================

def profile_dataframe(df, file_name):

    print("\nFirst 5 records:")
    df.show(5, truncate=False)

    print("\nTotal records:")

    row_count = df.count()

    print(row_count)

    print("\nTotal columns:")
    print(len(df.columns))

    print("\nColumns:")
    print(df.columns)

    print("\nData types:")
    print(df.dtypes)

    print("\nStatistical information:")
    df.describe().show(truncate=False)

    columns = df.columns

    if not columns:

        raise ValueError("Dataset has no columns")

    # Build a single aggregation for null and unique counts.

    expressions = []

    for index, column in enumerate(columns):

        safe_column = F.col(
            "`" + column.replace("`", "``") + "`"
        )

        expressions.append(
            F.count(
                F.when(safe_column.isNull(), 1)
            ).alias(f"null_{index}")
        )

        expressions.append(
            F.countDistinct(safe_column).alias(
                f"unique_{index}"
            )
        )

    profile_result = df.agg(*expressions).first()

    # Calculate duplicate records.

    distinct_count = df.dropDuplicates().count()

    duplicates = row_count - distinct_count

    print("\nDuplicate records:", duplicates)

    # Build column-level profiling metadata.

    profile_rows = []

    for index, column in enumerate(columns):

        null_count = profile_result[f"null_{index}"]

        unique_count = profile_result[f"unique_{index}"]

        null_percentage = (
            round((null_count / row_count) * 100, 2)
            if row_count > 0 else 0.0
        )

        data_type = dict(df.dtypes)[column]

        profile_rows.append((
            str(column),
            str(data_type),
            int(null_count),
            int(row_count),
            int(unique_count),
            float(null_percentage)
        ))

    profile_schema = StructType([

        StructField("column_name", StringType(), True),
        StructField("data_type", StringType(), True),
        StructField("null_count", LongType(), True),
        StructField("actual_count", LongType(), True),
        StructField("unique_count", LongType(), True),
        StructField("null_percentage", DoubleType(), True)

    ])

    column_profile = spark.createDataFrame(
        profile_rows,
        schema=profile_schema
    )

    print("\nColumn profile:")
    column_profile.show(truncate=False)

    # Save profile as a Spark CSV output directory.

    profile_file = OUTPUT / f"{Path(file_name).stem}_profile"

    (
        column_profile.write
        .mode("overwrite")
        .option("header", "true")
        .csv(str(profile_file))
    )

    print("\nProfile saved:", profile_file)

    data_types = dict(df.dtypes)

    return data_types, column_profile, duplicates


# ============================================================
# 5. BUILD GEMINI PROMPTS
# ============================================================

def build_prompts(file_name, df, data_types, column_profile):

    # Collect only 10 sample records.
    # The complete Spark DataFrame stays distributed.

    sample_records = [
        json.loads(record)
        for record in df.limit(10).toJSON().collect()
    ]

    sample_json = json.dumps(
        sample_records,
        indent=2,
        ensure_ascii=False
    )

    columns = df.columns

    types_json = json.dumps(
        data_types,
        indent=2,
        ensure_ascii=False
    )

    profile_rows = column_profile.collect()

    profile_text = "\n".join(
        [
            " | ".join(str(value) for value in row)
            for row in profile_rows
        ]
    )

    classification_prompt = f"""
You are a Data Engineering and Data Governance expert.

Analyze this dataset.

File: {file_name}

Columns: {json.dumps(columns)}

Spark data types:
{types_json}

Column profile:
{profile_text}

Sample records:
{sample_json}

Identify:

1. Business domain
2. Dataset type
3. PII columns
4. Sensitive columns
5. Identifier columns
6. Business measure columns
7. Date columns
8. Categorical columns

Explain the classification of every column.

Return valid JSON only, without Markdown fences.

Use only columns that exist in the dataset.

If a classification cannot be determined, state "unknown".

Do not infer sensitive personal attributes from names
or weak evidence.

Flag possible PII for human review.
"""

    schema_prompt = f"""
You are a Senior Data Engineer.

Analyze this source dataset.

File: {file_name}

Columns: {json.dumps(columns)}

Spark data types:
{types_json}

Column profile:
{profile_text}

Sample records:
{sample_json}

Determine:

1. Dataset business purpose
2. Recommended target table
3. Candidate primary key
4. Recommended partition column, if appropriate
5. Business meaning of every column
6. Recommended target data type
7. Nullable or not, based on evidence
8. Possible foreign keys
9. Data quality concerns

Return valid JSON only, without Markdown fences.

Use only columns that actually exist.

Do not claim a primary key or foreign key is confirmed
unless evidence establishes it.

Label uncertain recommendations clearly.
"""

    return classification_prompt, schema_prompt


# ============================================================
# 6. GEMINI API CALL
# ============================================================

def ask_gemini(prompt):

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response")

    return response.text


# ============================================================
# 7. PROCESS INDIVIDUAL FILE
# ============================================================

def process_file(source_file):

    file_name = source_file.name

    processing_file = PROCESSING / file_name

    print("\n======================================")
    print("Processing:", file_name)
    print("======================================")

    print(
        "File size:",
        source_file.stat().st_size,
        "bytes"
    )

    # Move source to processing.

    shutil.move(
        str(source_file),
        str(processing_file)
    )

    try:

        # Read using Spark.

        df = load_dataframe(processing_file)

        if df is None:
            raise ValueError("Unsupported file format")

        if len(df.columns) == 0:
            raise ValueError("Dataset has no columns")

        # Spark profiling.

        data_types, column_profile, duplicates = (
            profile_dataframe(df, file_name)
        )

        # Build prompts.

        classification_prompt, schema_prompt = build_prompts(
            file_name,
            df,
            data_types,
            column_profile
        )

        # Gemini classification.

        print("\nRequesting AI classification...")

        classification = ask_gemini(
            classification_prompt
        )

        # Gemini target schema recommendation.

        print("\nRequesting target schema recommendations...")

        schema = ask_gemini(
            schema_prompt
        )

        # Save AI results.

        classification_file = (
            OUTPUT /
            f"{Path(file_name).stem}_classification.json"
        )

        schema_file = (
            OUTPUT /
            f"{Path(file_name).stem}_schema.json"
        )

        classification_file.write_text(
            classification,
            encoding="utf-8"
        )

        schema_file.write_text(
            schema,
            encoding="utf-8"
        )

        # Save processed data using Spark.

        processed_file = (
            OUTPUT /
            f"{Path(file_name).stem}_processed"
        )

        (
            df.write
            .mode("overwrite")
            .option("header", "true")
            .csv(str(processed_file))
        )

        # Archive source file.

        shutil.move(
            str(processing_file),
            str(ARCHIVE / file_name)
        )

        print("\nSUCCESS:", file_name)

        print("Profile:", OUTPUT / f"{Path(file_name).stem}_profile")

        print("Classification:", classification_file)

        print("Schema:", schema_file)

        print("Processed data:", processed_file)

        print("Duplicate records:", duplicates)

        print("Source archived:", ARCHIVE / file_name)

    except Exception as e:

        print("\nERROR processing:", file_name)
        print("Error:", str(e))

        if processing_file.exists():

            shutil.move(
                str(processing_file),
                str(ERROR / file_name)
            )


# ============================================================
# 8. MAIN PIPELINE
# ============================================================

def main():

    supported_extensions = [
        ".csv",
        ".json",
        ".xlsx",
        ".txt"
    ]

    files = []

    for file_path in LANDING.iterdir():

        if file_path.is_file():

            if file_path.suffix.lower() in supported_extensions:

                files.append(file_path)

            else:

                print("\nUnsupported file:", file_path.name)

                shutil.move(
                    str(file_path),
                    str(ERROR / file_path.name)
                )

    files.sort()

    print("\n======================================")
    print("SPARK DATA ENGINEERING DATA PROFILER")
    print("Gemini model:", MODEL)
    print("======================================")

    print("\nFiles available in landing:")

    for file_path in files:
        print(file_path.name)

    if not files:

        print("\nNo files found in landing.")

        return

    for source_file in files:

        process_file(source_file)

    print("\n======================================")
    print("PIPELINE EXECUTION COMPLETED")
    print("======================================")

    print("Check:")
    print("data/output")
    print("data/archive")
    print("data/error")


# ============================================================
# 9. EXECUTE PIPELINE
# ============================================================

main()


from pathlib import Path
from dotenv import load_dotenv
from google import genai
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)


PROJECT_FOLDER = Path("/Volumes/workspace/default/projectsetup")
LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"

PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"

for folder in [LANDING, PROCESSING, ARCHIVE, ERROR, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_FOLDER / ".env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"


SPARK DATA ENGINEERING DATA PROFILER
Gemini model: gemini-3.5-flash-lite

Files available in landing:

No files found in landing.


In [0]:
def load_dataframe(file_path):

    extension = file_path.suffix.lower()
    file_path = str(file_path)

    if extension == ".csv":

        return (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("mode", "FAILFAST")
            .csv(file_path)
        )

    elif extension == ".json":

        return (
            spark.read
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .json(file_path)
        )

    elif extension == ".txt":

        return (
            spark.read
            .option("header", "true")
            .option("sep", "|")
            .option("inferSchema", "true")
            .option("mode", "FAILFAST")
            .csv(file_path)
        )

    elif extension == ".xlsx":

        # Excel requires an additional Spark connector
        # or a separate Excel reader.
        # Here Pandas is used only for ingestion.

        pandas_df = pd.read_excel(file_path)

        return spark.createDataFrame(pandas_df)

    else:

        print("Unsupported file type:", extension)
        return None

In [0]:
def profile_dataframe(df, file_name):

    print("\nFirst 5 records:")
    df.show(5, truncate=False)

    print("\nTotal records:")

    row_count = df.count()

    print(row_count)

    print("\nTotal columns:")
    print(len(df.columns))

    print("\nColumns:")
    print(df.columns)

    print("\nData types:")
    print(df.dtypes)

    print("\nStatistical information:")
    df.describe().show(truncate=False)

    columns = df.columns

    print(columns)

    if not columns:

        raise ValueError("Dataset has no columns")

    # Build a single aggregation for null and unique counts.

    expressions = []

    for index, column in enumerate(columns):

        safe_column = F.col(
            "`" + column.replace("`", "``") + "`"
        )

        expressions.append(
            F.count(
                F.when(safe_column.isNull(), 1)
            ).alias(f"null_{index}")
        )

        expressions.append(
            F.countDistinct(safe_column).alias(
                f"unique_{index}"
            )
        )
    
    print("expression is",expressions)


    profile_result = df.agg(*expressions).first()

    print("profile result",profile_result)



    # Calculate duplicate records.

    distinct_count = df.dropDuplicates().count()

    duplicates = row_count - distinct_count

    print("\nDuplicate records:", duplicates)

      # Build column-level profiling metadata.

    profile_rows = []

    for index, column in enumerate(columns):

        null_count = profile_result[f"null_{index}"]

        unique_count = profile_result[f"unique_{index}"]

        null_percentage = (
            round((null_count / row_count) * 100, 2)
            if row_count > 0 else 0.0
        )

        data_type = dict(df.dtypes)[column]

        profile_rows.append((
            str(column),
            str(data_type),
            int(null_count),
            int(row_count),
            int(unique_count),
            float(null_percentage)
        ))

    profile_schema = StructType([

        StructField("column_name", StringType(), True),
        StructField("data_type", StringType(), True),
        StructField("null_count", LongType(), True),
        StructField("actual_count", LongType(), True),
        StructField("unique_count", LongType(), True),
        StructField("null_percentage", DoubleType(), True)

    ])

    column_profile = spark.createDataFrame(
        profile_rows,
        schema=profile_schema
    )

    print("\nColumn profile:")
    column_profile.show(truncate=False)

    # Save profile as a Spark CSV output directory.

    profile_file = OUTPUT / f"{Path(file_name).stem}_profile"

    (
        column_profile.write
        .mode("overwrite")
        .option("header", "true")
        .csv(str(profile_file))
    )

    print("\nProfile saved:", profile_file)

    data_types = dict(df.dtypes)

    return data_types, column_profile, duplicates

In [0]:
# ============================================================
# 5. BUILD GEMINI PROMPTS
# ============================================================

def build_prompts(file_name, df, data_types, column_profile):

    # Collect only 10 sample records.
    # The complete Spark DataFrame stays distributed.

    """
    sample_records = [
        json.loads(record)
        for record in df.limit(10).toJSON().collect()
    ]

    sample_json = json.dumps(
        sample_records,
        indent=2,
        ensure_ascii=False
    )
    """
    sample_records = [
        row.asDict(recursive=True)
        for row in df.limit(10).collect()
    ]

    sample_json = json.dumps(
        sample_records,
        indent=2,
        ensure_ascii=False,
        default=str
    )


    columns = df.columns

    types_json = json.dumps(
        data_types,
        indent=2,
        ensure_ascii=False
    )

    profile_rows = column_profile.collect()

    profile_text = "\n".join(
        [
            " | ".join(str(value) for value in row)
            for row in profile_rows
        ]
    )

    classification_prompt = f"""
You are a Data Engineering and Data Governance expert.

Analyze this dataset.

File: {file_name}

Columns: {json.dumps(columns)}

Spark data types:
{types_json}

Column profile:
{profile_text}

Sample records:
{sample_json}

Identify:

1. Business domain
2. Dataset type
3. PII columns
4. Sensitive columns
5. Identifier columns
6. Business measure columns
7. Date columns
8. Categorical columns

Explain the classification of every column.

Return valid JSON only, without Markdown fences.

Use only columns that exist in the dataset.

If a classification cannot be determined, state "unknown".

Do not infer sensitive personal attributes from names
or weak evidence.

Flag possible PII for human review.
"""

    schema_prompt = f"""
You are a Senior Data Engineer.

Analyze this source dataset.

File: {file_name}

Columns: {json.dumps(columns)}

Spark data types:
{types_json}

Column profile:
{profile_text}

Sample records:
{sample_json}

Determine:

1. Dataset business purpose
2. Recommended target table
3. Candidate primary key
4. Recommended partition column, if appropriate
5. Business meaning of every column
6. Recommended target data type
7. Nullable or not, based on evidence
8. Possible foreign keys
9. Data quality concerns

Return valid JSON only, without Markdown fences.

Use only columns that actually exist.

Do not claim a primary key or foreign key is confirmed
unless evidence establishes it.

Label uncertain recommendations clearly.
"""

    return classification_prompt, schema_prompt


In [0]:
def ask_gemini(prompt):

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response")

    return response.text


In [0]:
def process_file(source_file):

    file_name = source_file.name

    processing_file = PROCESSING / file_name

    print("\n======================================")
    print("Processing:", file_name)
    print("======================================")

    print(
        "File size:",
        source_file.stat().st_size,
        "bytes"
    )

    # Move source to processing.

    
    #shutil.move(
      #  str(source_file),
       # str(processing_file)
    #)
    
    

    try:

        # Read using Spark.

        df = load_dataframe(processing_file)

        from pyspark import StorageLevel

        df = df.persist(StorageLevel.MEMORY_AND_DISK)

        if df is None:
            raise ValueError("Unsupported file format")

        if len(df.columns) == 0:
            raise ValueError("Dataset has no columns")

        data_types, column_profile, duplicates = (
            profile_dataframe(df, file_name)
        )

        classification_prompt, schema_prompt = build_prompts(
            file_name,
            df,
            data_types,
            column_profile
        )

        # Gemini classification.

        print("\nRequesting AI classification...")

        classification = ask_gemini(
            classification_prompt
        )

        # Gemini target schema recommendation.

        print("\nRequesting target schema recommendations...")

        schema = ask_gemini(
            schema_prompt
        )

        # Save AI results.

        classification_file = (
            OUTPUT /
            f"{Path(file_name).stem}_classification.json"
        )

        schema_file = (
            OUTPUT /
            f"{Path(file_name).stem}_schema.json"
        )

        classification_file.write_text(
            classification,
            encoding="utf-8"
        )

        schema_file.write_text(
            schema,
            encoding="utf-8"
        )

        # Save processed data using Spark.

        processed_file = (
            OUTPUT /
            f"{Path(file_name).stem}_processed"
        )

        (
            df.write
            .mode("overwrite")
            .option("header", "true")
            .csv(str(processed_file))
        )

        # Archive source file.

        shutil.move(
            str(processing_file),
            str(ARCHIVE / file_name)
        )

        print("\nSUCCESS:", file_name)

        print("Profile:", OUTPUT / f"{Path(file_name).stem}_profile")

        print("Classification:", classification_file)

        print("Schema:", schema_file)

        print("Processed data:", processed_file)

        print("Duplicate records:", duplicates)

        print("Source archived:", ARCHIVE / file_name)




    except Exception as e:

        print("\nERROR processing:", file_name)
        print("Error type:", type(e).__name__)
        print("Error details:", str(e))

        if processing_file.exists():

            shutil.move(
                str(processing_file),
                str(ERROR / file_name)
            )

        import traceback
        traceback.print_exc()


    
    finally:

        # =========================================
        # RELEASE SPARK CACHE
        # =========================================

        if df is not None:

            df.unpersist(blocking=True)

            print("\nSpark cache released.")


In [0]:
def main():

    supported_extensions = [
        ".csv",
        ".json",
        ".xlsx",
        ".txt"
    ]

    files = []

    for file_path in LANDING.iterdir():

        if file_path.is_file():

            if file_path.suffix.lower() in supported_extensions:

                files.append(file_path)

            else:

                print("\nUnsupported file:", file_path.name)

                shutil.move(
                    str(file_path),
                    str(ERROR / file_path.name)
                )

    files.sort()

    print("\n======================================")
    print("SPARK DATA ENGINEERING DATA PROFILER")
    print("Gemini model:", MODEL)
    print("======================================")

    print("\nFiles available in landing:")

    for file_path in files:
        print(file_path.name)

    if not files:

        print("\nNo files found in landing.")

        return
    
    for source_file in files:

        process_file(source_file)
    
main()


SPARK DATA ENGINEERING DATA PROFILER
Gemini model: gemini-3.5-flash-lite

Files available in landing:
cust.csv

Processing: cust.csv
File size: 6170902 bytes

First 5 records:
+---------------------------------------------------------+--------+----------+-----------+---------------------------+-----------+--------+---------------------+
|Address                                                  |BranchID|CustomerID|DateOfBirth|Email                      |FirstName  |LastName|PhoneNumber          |
+---------------------------------------------------------+--------+----------+-----------+---------------------------+-----------+--------+---------------------+
|605 Christina Run Suite 478 North Jane, MH 48812         |115     |443008    |2002-04-27 |christinecox@example.com   |Ryan       |Koch    |001-302-482-8337     |
|0747 Cassandra Way Kennethfurt, WV 81305                 |221     |443009    |1978-07-01 |montgomerykerri@example.net|Christopher|Rogers  |(892) 922-3348       |
|65997 T